# Experiment 2: The Eight Queens Problem

**Course:** CSAI4014 – Algorithms for Intelligent Systems and Robotics  
**Experiment:** 2

Run all cells from top to bottom in a fresh kernel. The notebook implements generate-and-test and backtracking with pruning, compares them for board sizes 4 through 10, and displays the first 8×8 solution.

## 1. Aim
To formulate the eight queens problem as a constraint satisfaction problem, to solve it by generate-and-test and by backtracking, and to measure the effect that the choice of representation and search strategy has on the cost of the solution.

## 2. Objectives
1. State a problem as variables, domains and constraints.
2. Show how the encoding removes whole classes of conflict before any search begins.
3. Implement generate-and-test and measure its cost.
4. Implement backtracking with pruning and count the nodes it visits.
5. Compare the two empirically over a range of board sizes.

## 3. Requirements
Python only. No third-party package is needed beyond the standard library, though Matplotlib is used to plot the timing comparison.

## 4. Theory
A constraint satisfaction problem consists of variables, a domain for each variable, and constraints. Represent each column as a variable and the row containing its queen as the value. This guarantees one queen per column. Using a permutation of row indices also guarantees one queen per row. Diagonal conflicts still need to be checked explicitly: two queens in columns i and j attack diagonally when |rᵢ − rⱼ| = |i − j|.

## 5. Implementation

In [ ]:
from itertools import permutations
import time
import matplotlib.pyplot as plt

def conflicts(board):
    \"\"\"board[c] = row of the queen in column c. Count attacking pairs.\"\"\"
    n = len(board)
    bad = 0
    for i in range(n):
        for j in range(i+1, n):
            if board[i] == board[j] or abs(board[i]-board[j]) == abs(i-j):
                bad += 1
    return bad

def generate_test(n):
    solutions = []
    for candidate in permutations(range(n)):
        if conflicts(candidate) == 0:
            solutions.append(candidate)
    return solutions

def is_safe(board, col, row):
    \"\"\"Check the proposed position against already-filled columns.\"\"\"
    for c in range(col):
        if board[c] == row or abs(board[c]-row) == abs(c-col):
            return False
    return True

def solve(n=8):
    solutions = []
    nodes = [0]

    def place(board, col):
        if col == n:
            solutions.append(tuple(board))
            return

        for row in range(n):
            nodes[0] += 1
            if is_safe(board, col, row):
                board[col] = row
                place(board, col+1)
                board[col] = -1  # undo placement before trying another row

    place([-1]*n, 0)
    return solutions, nodes[0]


## 6. Run both methods for n = 4 to 10

In [ ]:
results = []
for n in range(4, 11):
    start = time.perf_counter()
    backtracking_solutions, nodes = solve(n)
    backtracking_time = time.perf_counter() - start

    start = time.perf_counter()
    generate_solutions = generate_test(n)
    generate_time = time.perf_counter() - start

    # Both approaches must find the same number of solutions.
    assert len(backtracking_solutions) == len(generate_solutions)

    results.append((n, len(backtracking_solutions), nodes,
                    backtracking_time, generate_time))

print(f"{'n':>3} {'solutions':>10} {'nodes':>10} {'backtracking (s)':>18} {'generate-test (s)':>19}")
for n, count, nodes, tb, tg in results:
    print(f"{n:>3} {count:>10} {nodes:>10} {tb:>18.6f} {tg:>19.6f}")


## 7. First solution for an 8×8 board

In [ ]:
solutions_8, nodes_8 = solve(8)
first_solution = solutions_8[0]
print("Number of solutions for n=8:", len(solutions_8))
print("Nodes visited:", nodes_8)
print("First solution (row index for each column):", first_solution)

# Display the board. Q marks a queen and . marks an empty square.
for row in range(8):
    print(" ".join("Q" if first_solution[col] == row else "."
                   for col in range(8)))


## 8. Plot the execution-time comparison

In [ ]:
board_sizes = [row[0] for row in results]
backtracking_times = [row[3] for row in results]
generate_times = [row[4] for row in results]

plt.figure(figsize=(8, 4.5))
plt.plot(board_sizes, backtracking_times, "o-", label="Backtracking")
plt.plot(board_sizes, generate_times, "s-", label="Generate-and-test")
plt.yscale("log")
plt.xlabel("Board size (n)")
plt.ylabel("Execution time (seconds, logarithmic scale)")
plt.title("Execution time comparison")
plt.legend()
plt.figtext(0.5, 0.01, "Figure 2.1: Generate-and-test becomes increasingly slower as n grows.", ha="center")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()


## 9. Results and observations
Record, for each n, the number of solutions, nodes visited by backtracking, and execution time of both methods. Confirm that both methods report the same solution count. Record the first 8×8 solution as a board.

## 10. Discussion
Generate-and-test enumerates every permutation and checks it only after the complete candidate has been formed. Backtracking checks a queen against the columns already filled and rejects unsafe partial boards immediately. The node count and timing comparison show the benefit of pruning. Execution times depend on the computer and should be taken from the current run.

## 11. Applications
Backtracking with constraint propagation is used in timetabling, register allocation in compilers, Sudoku solvers and frequency assignment to transmitters. In robotics, similar constraints can appear in task allocation across a fleet of robots.

## 12. Conclusion
The eight queens problem was represented as a constraint satisfaction problem and solved using generate-and-test and backtracking. Both methods should return the same solution counts, while backtracking reduces unnecessary search by rejecting conflicting partial assignments.

## 13. Viva questions
1. Using a list of eight row indices, which conflicts become impossible by construction, and which do not?
2. Your solver reports 2113 solutions instead of 92. What single character is wrong, and where?
3. Why does `is_safe` examine only the columns to the left?
4. Backtracking is still exponential in the worst case. Why is it nonetheless the right choice here?
5. Explain why the 92 solutions reduce to 12 fundamental ones, and why the number is not 96.

## Source
CSAI4014 Algorithms for Intelligent Systems and Robotics Laboratory Manual, Experiment 2: The eight queens problem, Chapter 3, pages 16–19. Python standard library and Matplotlib are used for the implementation and plotting.